#Prepare Refined Datasets
#Cleaning, Removing Duplicates, Change Datatypes and perfprming joins

In [0]:
spark.sql(f" drop table if exists silver.shows")
spark.sql(f" drop table if exists silver.episodes")
spark.sql(f" drop table if exists silver.cast" )

In [0]:
spark.sql(f" drop schema if exists silver.shows")
spark.sql(f" drop schema if exists silver.episodes")
spark.sql(f" drop schema if exists silver.cast" )

In [0]:
%sql
SELECT COUNT(*) FROM bronze.shows;
SELECT COUNT(*) FROM bronze.episodes;
SELECT COUNT(*) FROM bronze.cast;

#Preapare Dataframe/tables

In [0]:
from pyspark.sql.functions import broadcast

df_shows    = spark.table("bronze.shows")
df_episodes = spark.table("bronze.episodes")
df_cast     = spark.table("bronze.cast")

In [0]:
df_shows.limit(5).display()

In [0]:
df_episodes.limit(5).display()

In [0]:
df_cast.limit(5).display()

#Getting Schemas of the Dataframes

In [0]:
df_shows.printSchema()

In [0]:
df_episodes.printSchema()

In [0]:
df_cast.printSchema()

In [0]:
%sql
DESCRIBE TABLE bronze.shows;

In [0]:
%sql
DESCRIBE TABLE bronze.episodes;

In [0]:
%sql
DESCRIBE TABLE bronze.cast;

Part 2: Transformations, PySpark Joins & Performance (Silver Layer)

In [0]:
from pyspark.sql.functions import explode_outer

from pyspark.sql.functions import col
from pyspark.sql.types import StructType, ArrayType


def flatten_structs(df):
    complex_fields = True

    while complex_fields:
        complex_fields = False
        for field in df.schema.fields:
            if isinstance(field.dataType, StructType):
                complex_fields = True

                expanded_cols = [
                    col(f"{field.name}.{nested.name}")
                    .alias(f"{field.name}_{nested.name}")
                    for nested in field.dataType.fields
                ]

                df = df.select("*", *expanded_cols).drop(field.name)

    return df


In [0]:
from pyspark.sql.functions import explode_outer


def explode_arrays(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType):
            df = df.withColumn(field.name, explode_outer(col(field.name)))
    return df

In [0]:
def explode_and_flatten(df):
    while True:
        array_fields = [
            field.name for field in df.schema.fields
            if isinstance(field.dataType, ArrayType)
        ]
        struct_fields = [
            field.name for field in df.schema.fields
            if isinstance(field.dataType, StructType)
        ]

        if not array_fields and not struct_fields:
            break

        if array_fields:
            df = explode_arrays(df)

        if struct_fields:
            df = flatten_structs(df)

    return df

In [0]:

df_silver = explode_and_flatten(df_shows)

df_silver.printSchema()
df_silver.display()

Creating Schema for Silver DB

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.shows")

In [0]:
df_episodes = explode_and_flatten(df_episodes)

df_episodes.printSchema()
df_episodes.display()

In [0]:
df_episodes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.episodes")

In [0]:
df_cast = explode_and_flatten(df_cast)

df_cast.printSchema()
df_cast.display()

In [0]:
df_cast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.cast")

Count Null values for all coulmns in a DataFrame


In [0]:

from pyspark.sql.functions import count, when, col

df_shows.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_shows.columns
]).display()

In [0]:

from pyspark.sql.functions import count, when, col

df_episodes.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_episodes.columns
]).display()

In [0]:

from pyspark.sql.functions import count, when, col

df_cast.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_cast.columns
]).display()

Replacing missing values with nothing("") if it is String column
and zero(0) If it is float or decimal or INT

In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import (
    StringType, IntegerType, LongType,
    DoubleType, FloatType, DecimalType, ShortType
)

def replace_nulls_by_datatype(df):
    for field in df.schema.fields:
        col_name = field.name
        data_type = field.dataType

        # String columns → empty string
        if isinstance(data_type, StringType):
            df = df.withColumn(
                col_name,
                when(col(col_name).isNull(), "").otherwise(col(col_name))
            )

        # Numeric columns → 0
        elif isinstance(
            data_type,
            (IntegerType, LongType, DoubleType, FloatType, DecimalType, ShortType)
        ):
            df = df.withColumn(
                col_name,
                when(col(col_name).isNull(), 0).otherwise(col(col_name))
            )

    return df

In [0]:
from pyspark.sql.functions import broadcast

df_shows    = spark.table("silver.shows")
df_episodes = spark.table("silver.episodes")
df_cast     = spark.table("silver.cast")

In [0]:
df_shows_clean = replace_nulls_by_datatype(df_shows)

In [0]:
df_episodes_clean = replace_nulls_by_datatype(df_episodes)

In [0]:
df_cast_clean = replace_nulls_by_datatype(df_cast)

In [0]:
from pyspark.sql.functions import count, when, col

df_shows_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_shows_clean.columns
]).display()

In [0]:
from pyspark.sql.functions import count, when, col

df_episodes_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_episodes_clean.columns
]).display()

In [0]:
from pyspark.sql.functions import count, when, col

df_cast_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_cast_clean.columns
]).display()

#Creating Final DataFrame/table Fact table 

In [0]:
from pyspark.sql.functions import broadcast, col

fact_df = (
    df_shows_clean.alias("s")
    .join(df_episodes_clean.alias("e"), col("s.id") == col("e.id"), "inner")
    .join(
        broadcast(df_cast_clean.alias("c")),
        col("s.id") == col("c.show_id"),
        "left"
    )
    .select(
        col("s.id"),
        col("s.name").alias("show_name"),
        col("s.language"),
        col("s.genres").alias("genre"),
        col("e.id").alias("episode_id"),
        col("e.name").alias("episode_name"),
        col("e.season").alias("season"),
        col("e.airdate"),
        col("e.runtime"),
        col("c.person_name").alias("cast_name"),
        col("c.character_name")
    )
)


In [0]:
display(fact_df)

In [0]:
%python
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
#Delta Optimization process options
#Set max records per file for optimize file size
spark.conf.set("spark.sql.files.maxPartitionBytes", 1000000)
#Set parallelism for better concurrency
spark.conf.set("spark.sql.shuffle.partitions", 200)
#Set Auto optimise the Delta table
# spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", True)
# spark.conf.set("spark.databricks.delta.autoOptimize.optimizeWrite", True)
# spark.conf.set("spark.sql.adaptive.enabled","true")
# spark.conf.set("spark.sql.adaptive.join.enabled","true")
# spark.conf.set("spark.databricks.delta.autoCompact.enabled","true")
# spark.conf.set("spark.sql.adaptive.skewJoin.enabled","true")


In [0]:
## Writing the output data in a specified folder
fact_df.coalesce(5).write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.option("mergeSchema", "True") \
.option("compression", "zstd") \
.saveAsTable("silver.fact_table")


In [0]:
%sql
select count(*) from silver.fact_table

In [0]:
spark.sql(f"""
    DESCRIBE DETAIL silver.fact_table
""").display()

Write With Partitioning

In [0]:
fact_df.coalesce(5).write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.option("mergeSchema", "True") \
.option("compression", "zstd") \
.partitionBy("language") \
.saveAsTable("silver.fact_table_partitioned")

Broadcast Join Implementation (Mandatory Demonstration)
Broadcast shows_df

In [0]:
from pyspark.sql.functions import (
    broadcast, rand, floor, explode, array, lit
)

In [0]:
from pyspark.sql.functions import broadcast

broadcast_join_df = (
    df_episodes
    .join(broadcast(df_shows), "id", "inner")
    .join(df_cast, "show_id", "inner")
)

Execution Plan Validation (Must Show)

In [0]:
broadcast_join_df.explain(True) 

Broadcast Multiple Tables 

In [0]:
fully_broadcast_df = (
    df_episodes
    .join(broadcast(df_shows), "id", "inner")
    .join(broadcast(df_cast), "show_id", "inner")
)

In [0]:
fully_broadcast_df.count()

Step by Step Salting Implementation
Define Number of Salt Buckets

In [0]:
SALT_BUCKETS = 10

Add Salt to Large Tables
Add random salt to episodes_df and cast_df

In [0]:
from pyspark.sql.functions import rand, floor

episodes_salted = df_episodes.withColumn(
    "salt",
    floor(rand() * SALT_BUCKETS)
)

cast_salted = df_cast.withColumn(
    "salt",
    floor(rand() * SALT_BUCKETS)
)

In [0]:
from pyspark.sql.functions import explode, array, lit

shows_salted = (
    df_shows
    .withColumn(
        "salt",
        explode(array([lit(i) for i in range(SALT_BUCKETS)]))
    )
)

Perform the Salted Join

In [0]:
salted_join_df = (
    episodes_salted
    .join(shows_salted, ["id", "salt"], "inner")
    .join(cast_salted, ["show_id", "salt"], "inner")
)

In [0]:
display(salted_join_df) 

In [0]:
salted_join_df.explain(True) 

Broadcast join timing

In [0]:
import time

start = time.time()

fully_broadcast_df.count()

print("Broadcast Join Time",time.time() - start)


Salted join timing

In [0]:
start = time.time()

salted_join_df.count()

print("Salted Join Time:", time.time() - start)

Apply OPTIMIZE + ZORDER

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS Optimize")

In [0]:
df_shows.write.format("delta").mode("overwrite").saveAsTable("Optimize.shows")
df_episodes.write.format("delta").mode("overwrite").saveAsTable("Optimize.episodes")
df_cast.write.format("delta").mode("overwrite").saveAsTable("Optimize.cast")

In [0]:
%sql

OPTIMIZE Optimize.shows
ZORDER BY (id)

In [0]:
%sql

OPTIMIZE Optimize.episodes
ZORDER BY (id, season)


In [0]:
%sql

OPTIMIZE  Optimize.cast
ZORDER BY (show_id, character_name)

How to Check Performance Improvement
File Count Reduction

In [0]:
%sql

SELECT *
FROM silver.episodes
WHERE id = '123'

In [0]:
%sql

explain SELECT *
FROM Optimize.episodes
WHERE id = '123'

In [0]:
display(spark.sql("DESCRIBE DETAIL silver.episodes"))

In [0]:
display(spark.sql("DESCRIBE DETAIL optimize.episodes"))


In [0]:

display(spark.sql("""
OPTIMIZE silver.episodes
ZORDER BY (id)
"""))
